# Classify objects from measured features

**Purpose.** Fit a supervised tabular classifier to per-object measurements and apply it to the full dataset.

**Recommended use.** Use when morphology, intensity, or spatial measurements provide an interpretable representation of the phenotype.

**Primary outputs.** Per-object predictions, validation metrics, feature importance, and SHAP summaries when supported.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.ml.ml_analysis`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.ml_analysis)

```python
ml_analysis(**settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.ml import ml_analysis

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.ml.ml_analysis`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.ml_analysis)


#### Measurements

- **`channel_of_interest`** *(optional)* — Channel index used to select features.
- **`exclude`** *(optional)* — Columns to remove from feature space.
- **`remove_highly_correlated_features`** *(optional)* — Drop highly correlated features.
- **`remove_low_variance_features`** *(optional)* — Drop low-variance features.

#### Plate Layout & Controls

- **`location_column`** *(optional)* — Column identifying wells / plate columns. Default ``'columnID'``.
- **`positive_control`** *(optional)* — Value(s) in ``location_column`` treated as the positive class. Default ``'c2'``.
- **`negative_control`** *(optional)* — Value(s) treated as the negative class. Default ``'c1'``.
- **`batch_correction`** *(optional)* — plate correction method from :mod:`spacr.batch_correction`.
- **`batch_column`** *(optional)* — metadata column identifying plates/batches.
- **`batch_control_column`** *(optional)* — metadata column holding reference-control labels for ``control_center``.
- **`batch_control_values`** *(optional)* — negative/reference control value(s).
- **`batch_covariate_column`** *(optional)* — metadata column naming the BIOLOGY the correction must protect -- treatment, cell line, timepoint. Only ``combat`` uses it, and for combat it is not optional: the covariate coefficients are kept while the batch ones are subtracted, so a contrast left out of the design lands in the batch term and is removed along with it. Omitting it is how a real effect gets "corrected" away.
- **`batch_combat_mean_only`** *(optional)* — adjust each batch's MEAN and leave its variance alone. Use it when a plate is shifted but not differently scaled, or when a batch has too few rows for a stable variance estimate -- the shrunken scale term is the part that goes wrong on small batches. Default False, which adjusts both.
- **`batch_min_samples`** *(optional)* — minimum rows or controls per plate.
- **`batch_missing_control`** *(optional)* — ``error`` or ``skip`` for missing controls.

#### Computer Vision Model

- **`model_type`** *(optional)* — ``'random_forest'``, ``'logistic_regression'``, ``'gradient_boosting'`` or ``'xgboost'``.

#### Computer Vision Training

- **`learning_rate`** *(optional)* — XGBoost learning rate.

#### Machine Learning Model and Features

- **`n_estimators`** *(optional)* — Tree count for tree-based models.
- **`test_size`** *(optional)* — Test-split fraction. Default ``0.2``.
- **`cross_validation`** *(optional)* — If True, run 5-fold stratified CV.
- **`reg_lambda`** *(optional)* — XGBoost L2 penalty.
- **`reg_alpha`** *(optional)* — XGBoost L1 penalty.
- **`prune_features`** *(optional)* — If True, apply ``SelectKBest`` before training.
- **`top_features`** *(optional)* — Feature cap when ``prune_features=True``.
- **`n_repeats`** *(optional)* — Repeats for permutation importance. Default ``10``.

#### Advanced

- **`verbose`** *(optional)* — Log progress details.
- **`n_jobs`** *(optional)* — Parallel job count where applicable. Default ``-1``.

#### Additional settings

- **`df`** *(required)* — Per-object feature DataFrame as produced by merging the cell/nucleus/pathogen/cytoplasm tables of a :func:`spacr.measure.measure_crop` database.
- **`split_by`** *(optional)* — Independent acquisition unit for train/test splitting: ``'cell'``, ``'field'``, ``'well'`` (default), or ``'plate'``. Legacy ``'none'`` is an alias for ``'cell'``.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Measurements
    # Optional settings
    'channel_of_interest': 3,
    'exclude': None,
    'remove_highly_correlated_features': True,
    'remove_low_variance_features': True,

    # Plate Layout & Controls
    # Optional settings
    'location_column': 'columnID',
    'positive_control': 'c2',
    'negative_control': 'c1',
    'batch_correction': 'none',
    'batch_column': 'plateID',
    'batch_control_column': None,
    'batch_control_values': None,
    'batch_covariate_column': None,
    'batch_combat_mean_only': False,
    'batch_min_samples': 3,
    'batch_missing_control': 'error',

    # Computer Vision Model
    # Optional settings
    'model_type': 'xgboost',

    # Computer Vision Training
    # Optional settings
    'learning_rate': 1e-05,

    # Machine Learning Model and Features
    # Optional settings
    'n_estimators': 1000,
    'test_size': 0.2,
    'cross_validation': False,
    'reg_lambda': 1.0,
    'reg_alpha': 0.1,
    'prune_features': False,
    'top_features': 30,
    'n_repeats': 10,

    # Advanced
    # Optional settings
    'verbose': False,
    'n_jobs': -1,

    # Additional settings
    # Required settings
    'df': None,
    # Optional settings
    'split_by': 'well',
}

In [ ]:
ml_analysis(settings)

## Outputs and next steps

Per-object predictions, validation metrics, feature importance, and SHAP summaries when supported.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)